In [ ]:
import pandas as pd
from pathlib import Path
import json

# --- Configuration ---
MICRONS_PER_PIXEL = 17.54  # Scaling factor: 17.54 microns per pixel
IMAGE_HEIGHT_PIXELS = 2052  # Full Y pixel dimension of the image

# Load the dataset index
project_root = Path.cwd().parent
processed_data_dir = project_root / 'data' / 'processed'
dataset_index_path = processed_data_dir / 'dataset_index.parquet'
dataset_index = pd.read_parquet(dataset_index_path)

# --- Process Geometry Data ---
geometry_data = []

# Define a mapping from label ID to its role
label_role_mapping = {
    0: 'ring_outer',
    1: 'ring_inner',
    2: 'plate_boundary'
}

for _, row in dataset_index.iterrows():
    labels_file_path = Path(row['labels_path'])
    if labels_file_path.exists():
        # It's a CSV file with label information
        labels_df = pd.read_csv(labels_file_path)
        
        for _, label_row in labels_df.iterrows():
            # Corrected column name from 'Label ID' to 'Label Id'
            label_id = label_row['Label Id'] 
            
            if label_id in label_role_mapping:
                role = label_role_mapping[label_id]
                
                # The geometry data is in the 'Region Properties' column
                try:
                    # Split the string "X, Y, Diameter" into parts
                    properties = str(label_row['Region Properties']).split(',')
                    if len(properties) != 3:
                        continue # Skip if the format is wrong
                    
                    bb_x, bb_y, diameter = map(float, properties)
                    
                    # Calculate center and radius from bounding box and diameter
                    radius_pixels = diameter / 2
                    cx_pixels = bb_x + radius_pixels
                    cy_pixels = bb_y + radius_pixels

                    # Transformation logic
                    cx_um = cx_pixels * MICRONS_PER_PIXEL
                    cy_um = (IMAGE_HEIGHT_PIXELS - cy_pixels) * MICRONS_PER_PIXEL
                    r_um = radius_pixels * MICRONS_PER_PIXEL
                    
                    geometry_data.append({
                        'dataset_id': row['dataset_id'],
                        'boundary_role': role,
                        'cx_um': cx_um,
                        'cy_um': cy_um,
                        'r_um': r_um
                    })
                except (ValueError, TypeError) as e:
                    print(f"Could not parse Region Properties for {row['dataset_id']}, Label Id {label_id}: {e}")
                    continue

if geometry_data:
    geometry_df = pd.DataFrame(geometry_data)
    geometry_output_path = processed_data_dir / 'geometry.parquet'
    geometry_df.to_parquet(geometry_output_path, index=False)
    print(f"Geometry data saved to {geometry_output_path}")
    print(geometry_df.head())
else:
    print("No geometry data found. Check the structure of your 'Labels.csv' files.")

In [ ]:
import numpy as np

# --- Process Tracks Data ---

def calculate_speed(df, time_gap_threshold=1.0):
    """Calculates gap-aware speed for each track."""
    df = df.sort_values('t_s')
    
    # Calculate time difference and displacement
    df['dt'] = df['t_s'].diff()
    df['dx'] = df['x_um'].diff()
    df['dy'] = df['y_um'].diff()
    
    # Identify time gaps
    is_gap = (df['dt'] > time_gap_threshold) | (df['dt'].isna())
    
    # Calculate speed, setting to NaN at gaps
    df['speed_um_s'] = np.sqrt(df['dx']**2 + df['dy']**2) / df['dt']
    df.loc[is_gap, 'speed_um_s'] = np.nan
    
    return df.drop(columns=['dt', 'dx', 'dy'])

def process_wide_format_file(file_path, value_name):
    """Reads a wide-format CSV, melts it to a long format, and cleans it up."""
    df = pd.read_csv(file_path, skiprows=1)
    df = df.rename(columns={'Time': 't_s'})
    
    # Melt the dataframe
    id_vars = ['Frame', 't_s']
    value_vars = [col for col in df.columns if col not in id_vars]
    long_df = df.melt(id_vars=id_vars, value_vars=value_vars, var_name='track_id', value_name=value_name)
    
    # Clean up track_id (remove spaces and suffixes)
    long_df['track_id'] = long_df['track_id'].str.extract('(\d+)').astype(int)
    
    # Drop rows with NaN values that result from melting empty cells
    long_df = long_df.dropna(subset=[value_name])
    
    return long_df.drop(columns=['Frame'])

tracks_data = []

for _, row in dataset_index.iterrows():
    position_path = Path(row['position_path'])
    area_path = Path(row['area_path'])
    fit_path = Path(row['fit_path'])
    length_path = Path(row['length_path'])
    
    if all([position_path.exists(), area_path.exists(), fit_path.exists(), length_path.exists()]):
        # 1. Process Position Data
        pos_df_raw = pd.read_csv(position_path, skiprows=1)
        pos_df_raw = pos_df_raw.rename(columns={'Time': 't_s'})
        pos_melted = pos_df_raw.melt(id_vars=['Frame', 't_s'], var_name='track_info', value_name='position')
        pos_melted['track_id'] = pos_melted['track_info'].str.extract('(\d+)').astype(int)
        pos_melted['axis'] = pos_melted['track_info'].str.extract('([xy])')
        pos_long = pos_melted.pivot_table(index=['t_s', 'track_id'], columns='axis', values='position').reset_index()
        pos_long = pos_long.rename(columns={'x': 'x_um', 'y': 'y_um'})
        pos_long = pos_long.dropna(subset=['x_um', 'y_um'])

        # 2. Process Measurement Data
        area_long = process_wide_format_file(area_path, 'area')
        fit_long = process_wide_format_file(fit_path, 'fit')
        length_long = process_wide_format_file(length_path, 'length')

        # 3. Merge Dataframes
        merged_df = pd.merge(pos_long, area_long, on=['t_s', 'track_id'], how='left')
        merged_df = pd.merge(merged_df, fit_long, on=['t_s', 'track_id'], how='left')
        merged_df = pd.merge(merged_df, length_long, on=['t_s', 'track_id'], how='left')
        
        # Add dataset_id for this file
        merged_df['dataset_id'] = row['dataset_id']
        
        tracks_data.append(merged_df)

if tracks_data:
    # --- Perform expensive operations once on the full dataframe ---
    
    # 1. Concatenate all data into a single dataframe
    full_tracks_df = pd.concat(tracks_data, ignore_index=True)

    # 2. Ensure correct data types before calculation
    full_tracks_df[['x_um', 'y_um']] = full_tracks_df[['x_um', 'y_um']].astype(float)
    
    # 3. Calculate speed (vectorized)
    # The groupby is now on a composite key to handle tracks from different datasets
    final_df = full_tracks_df.groupby(['dataset_id', 'track_id']).apply(calculate_speed).reset_index(drop=True)
    
    # 4. Create track_uid
    final_df['track_uid'] = final_df['dataset_id'].astype(str) + '_' + final_df['track_id'].astype(str)
    
    # 5. Reorder columns
    final_df = final_df[[
        'dataset_id', 'track_id', 'track_uid', 't_s', 'x_um', 'y_um',
        'fit', 'length', 'area', 'speed_um_s'
    ]]

    # 6. Save to Parquet
    tracks_output_path = processed_data_dir / 'tracks.parquet'
    final_df.to_parquet(tracks_output_path, index=False)
    print(f"Tracks data saved to {tracks_output_path}")
    print(final_df.head())
else:
    print("No tracks data found.")